# Gemini API: Analyze a Video - Historic Event Recognition

This notebook shows how you can use Gemini models' multimodal capabilities to recognize which historic event is happening in the video.

In [1]:
%pip install -U -q "google-genai>=1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


In [2]:
%pip install -U -q "google-genai>=1.0.0"


## Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.

In [3]:
#from google import genai
#from google.colab import userdata

#API_KEY = userdata.get('GOOGLE_API_KEY')
#client = genai.Client(api_key=API_KEY)

In [4]:
from google import genai
from google.colab import userdata
API_KEY=userdata.get('GOOGLE_API_KEY2')
client=genai.Client(api_key=API_KEY)


## Example

This example uses [video of Corneliu Zelea in the Romanian Parliament in 1933](https://www.youtube.com/watch?v=xBU0hE_o04Y).

In [5]:
# Download video
#path = "berlin.mp4"
#url = "https://s3.amazonaws.com/NARAprodstorage/opastorage/live/16/147/6014716/content/presidential-libraries/reagan/5730544/6-12-1987-439.mp4"
#!wget $url -O $path

In [6]:
import os

!pip install yt-dlp


# Download video using yt-dlp. It will save the file to /content/ with its original name.
!yt-dlp https://www.youtube.com/watch?v=xBU0hE_o04Y

# Define the original downloaded file name (as yt-dlp saves it)
original_downloaded_name = "Discurs Corneliu Zelea Codreanu 1933 (Declarația din parlament). [xBU0hE_o04Y].mp4"
original_path = os.path.join("/content", original_downloaded_name)

# Define a simpler, ASCII-only path for the video
new_path_name = "codreanu_speech.mp4"
new_path = os.path.join("/content", new_path_name)

# Rename the file to avoid issues with special characters during upload
# Check if the original file exists before renaming
if os.path.exists(original_path):
  !mv "$original_path" "$new_path"
  print(f"Renamed '{original_downloaded_name}' to '{new_path_name}'")
else:
  print(f"Original file '{original_downloaded_name}' not found. Assuming it was already renamed or not downloaded.")

# Set the 'path' variable to the new, simplified filename for subsequent cells
path = new_path
print(f"'path' variable set to: {path}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.4 MB/s eta 0:00:00
[youtube] Extracting URL: https://www.youtube.com/watch?v=xBU0hE_o04Y
[youtube] xBU0hE_o04Y: Downloading webpage
[youtube] xBU0hE_o04Y: Downloading android vr player API JSON
[info] xBU0hE_o04Y: Downloading 1 format(s): 18
[download] Destination: Discurs Corneliu Zelea Codreanu 1933 (Declarația din parlament). [xBU0hE_o04Y].mp4
[download] 100% of   17.40MiB in 00:00:01 at 10.84MiB/s
Renamed 'Discurs Corneliu Zelea Codreanu 1933 (Declarația din parlament). [xBU0hE_o04Y].mp4' to 'codreanu_speech.mp4'
'path' variable set to: /content/codreanu_speech.mp4


In [7]:
#Upload video
js_runtimes:{'deno':{'path':None},'node':{'path':'C:/Program Files/nodejs/node.exe'}}
video_file=client.files.upload(file=path)

In [8]:
import time
# Wait until the uploaded video is available
while video_file.state.name == "PROCESSING":
  print('.', end='')
  time.sleep(5)
  video_file = client.files.get(name=video_file.name)

if video_file.state.name == "FAILED":
  raise ValueError(video_file.state.name)

...

The uploaded video is ready for processing. This prompt instructs the model to provide basic information about the historical events portrayed in the video.

In [9]:
system_prompt = """
  You are historian who specializes in events caught on film.
  When you receive a video answer following questions:
  When did it happen?
  Who is the most important person in video?
  How the event is called?
"""

Some historic events touch on controversial topics that may get flagged by Gemini API, which blocks the response for the query.

Because of this, it might be a good idea to turn off safety settings.

In [10]:
safety_settings = [
    {
        "category": "HARM_CATEGORY_HARASSMENT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_HATE_SPEECH",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE",
    },
]

In [11]:
from google.genai import types

MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}
response = client.models.generate_content(
    model=f"models/{MODEL_ID}",
    contents=[
        "Analyze the video please",
        video_file
        ],
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        safety_settings=safety_settings,
        ),
    )
print(response.text)

Sure! Here are some details from the video:

- When did it happen? - The event happened in 1933.


- Who is the most important person in the video? - The most important person in the video is Corneliu Zelea Codreanu, Romanian politician of the far right, the founder and charismatic leader of the Iron Guard.


- How the event is called? - The event is called "The Declaration of 1933".


In [12]:
pip install moviepy transformers accelerate soundfile

In [13]:
import os
from moviepy.editor import VideoFileClip
from transformers import pipeline

# Define the output audio file name
audio_output_path = "audio.wav"

# Load the video clip from the previously defined 'path'
video_clip = VideoFileClip(path)

# Extract the audio
audio_clip = video_clip.audio

# Write the audio to a WAV file
audio_clip.write_audiofile(audio_output_path)

print(f"Audio extracted to: {audio_output_path}")

# Close the video and audio clips
audio_clip.close()
video_clip.close()

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



MoviePy - Writing audio in audio.wav


MoviePy - Done.
Audio extracted to: audio.wav


In [14]:
# Initialize the ASR pipeline using the Whisper small model
# You can choose a larger model like "openai/whisper-base" or "openai/whisper-medium" for better accuracy if needed,
# but it will require more memory and processing time.
asr_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-small")

# Transcribe the extracted audio file, enabling timestamp generation for long audio
transcript = asr_pipeline(audio_output_path, return_timestamps=True)



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

In [15]:
# Print the full transcript
print("Video Transcript:")
print(transcript['text'])

Video Transcript:
 Noi așteptăm un alt regim, un alt sistem care va veni, după ce pe acesta îl vor prămuși greutatea și mulțimea păcate lor lui. El trebuie să correspondă următoare lor cerinți în ordinea urgenței. 1. Să desfințeze aceste discuți sterile și scump plătite ale parlamentarismului democratic, din care n-a ieșit niciodată lumină și din care, mai ales, nu poate ieși hotărârea eroică de a face față primeștiei în ceasurile grele de acum. Doi, să se înlocuiască prin comandă care să adun într-un singur mânunchi toate energieiile disparate ale neamului, încleștate astăzi în luptă afratricidă, să le disciplineze, să le refacă moralul pierdut, Să le insufle credința în destinul neamului nostru românesc și să le conducă pe căile acestui destinii. 3. Să declareră zboi mizeriei și sărăciei generale, îmdemnând la muncă și cu împătare pe cei buni, trimitând cu forța la muncă toate elementele parazitare care joacă în stat rolul trântorilor din stup, Pe toți lene și care păzesc mesele ca f

## Summary

Now you know how you can prompt Gemini models with videos and use them to recognize historic events.

This notebook shows only one of many use cases. Check the [Video understanding](../quickstarts/Video_understanding.ipynb) notebook for more examples of using the Gemini API with videos.

In [12]:
# Delete video
client.files.delete(name=video_file.name)

DeleteFileResponse(
  sdk_http_response=HttpResponse(
    headers=<dict len=10>
  )
)

In [28]:
# Clean up the generated audio file after transcription (optional)
import os
if os.path.exists(audio_output_path):
    os.remove(audio_output_path)
    print(f"Removed temporary audio file: {audio_output_path}")

Removed temporary audio file: audio.wav
